# Phase 6A — Kaggle supervised-training smoke

**Prepared, not run.** Narration training only. Goal: verify targets and decoder policy, FP16 AMP loss/gradients, a real optimizer update, and a resumable checkpoint with optimizer, scheduler, scaler and RNG. This is **not** Phase 6B pilot training and reports no CER.

Attach the private **Phase 6A private training-smoke bundle (`phase6a_training_smoke_bundle_v1.zip`)** as a Kaggle input. Select a **Tesla T4 GPU** and enable Internet for the pinned public model. Keep this notebook and its output private; the bundle contains transcripts and IDs. Run cell 1 first. If it installs packages, restart the Kaggle session, then Run All. A successful run writes `/kaggle/working/phase6a_private/` (checkpoint and report). Do not commit or publish that folder.


In [ ]:
import importlib.metadata as im, subprocess, sys
PINS = {'transformers':'4.57.1','tokenizers':'0.22.2','huggingface-hub':'0.36.2','safetensors':'0.8.0'}
installed = {name: im.version(name) if any(d.metadata['Name'].lower()==name for d in im.distributions()) else None for name in PINS}
print('Installed Hugging Face packages:', installed)
if installed != PINS:
    cmd=[sys.executable,'-m','pip','install','--no-cache-dir','--no-deps',*[f'{k}=={v}' for k,v in PINS.items()]]
    subprocess.run(cmd,check=True)
    raise SystemExit('Pinned packages installed. RESTART the Kaggle session, then Run All from the top. PyTorch/CUDA was not changed.')
if 'transformers' in sys.modules and getattr(sys.modules['transformers'],'__version__',None) != PINS['transformers']:
    raise SystemExit('Restart the Kaggle session: an older Transformers was imported already.')


## 1. Verify runtime and training-only input

Use exactly one attached Phase 6A bundle. The notebook discovers it by marker, hashes all 12 WAVs, checks every reference against its tokenizer-target table, and selects eight different training components, including the shortest and near-30-second clips. No validation or test bundle is loaded.


In [ ]:
import csv, hashlib, json, math, os, platform, random, re, time, unicodedata
from pathlib import Path
import numpy as np
import soundfile as sf
import torch, transformers, tokenizers, huggingface_hub, safetensors
from transformers import WhisperProcessor, WhisperForConditionalGeneration
assert transformers.__version__=='4.57.1' and tokenizers.__version__=='0.22.2'
assert huggingface_hub.__version__=='0.36.2' and safetensors.__version__=='0.8.0'
assert torch.__version__.startswith('2.11.0'), f'Expected the measured Kaggle torch 2.11.0 build; found {torch.__version__}'
assert torch.cuda.is_available(), 'Enable a Kaggle Tesla T4 GPU and restart.'
assert 'T4' in torch.cuda.get_device_name(0), f'T4 expected; found {torch.cuda.get_device_name(0)}'
DEVICE=torch.device('cuda:0'); SEED=0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__,
       'tokenizers':tokenizers.__version__,'hub':huggingface_hub.__version__,
       'safetensors':safetensors.__version__,'numpy':np.__version__,'soundfile':sf.__version__,
       'gpu':torch.cuda.get_device_name(0),'seed':SEED})
roots=[]
for marker in Path('/kaggle/input').rglob('bundle_record.json'):
    try: rec=json.loads(marker.read_text())
    except (OSError,ValueError): continue
    if rec.get('bundle_type')=='phase6a_training_smoke_v1' and (marker.parent/'train_smoke_manifest.csv').is_file():
        roots.append((marker.parent.resolve(),rec))
assert len(roots)==1, f'Attach exactly one extracted Phase 6A training-smoke bundle; found {len(roots)}.'
BUNDLE, BUNDLE_RECORD=roots[0]
assert BUNDLE_RECORD.get('contains_validation_or_test_audio') is False
with (BUNDLE/'train_smoke_manifest.csv').open(encoding='utf-8',newline='') as f: smoke=list(csv.DictReader(f))
assert BUNDLE_RECORD.get('raw_and_normalized_reference_views') is True and BUNDLE_RECORD.get('frozen_train_membership_checked') is True
sums=(BUNDLE/'SHA256SUMS.txt').read_text().splitlines()
assert len(sums)==14, 'Expected twelve audio hashes plus manifest and bundle record.'
for line in sums:
    expected,relative=line.split('  ',1)
    path=(BUNDLE/relative).resolve(); path.relative_to(BUNDLE)
    assert hashlib.sha256(path.read_bytes()).hexdigest()==expected, f'Bundle hash mismatch: {relative}'
assert len(smoke)==12 and len({r['sample_id'] for r in smoke})==12
assert len({r['component_id'] for r in smoke})==12, 'Expected one training component per smoke clip.'
def normalize_raw_reference(text): return re.sub(r' {2,}',' ',unicodedata.normalize('NFD',text).strip())
def safe_path(relative):
    path=(BUNDLE/relative).resolve(); path.relative_to(BUNDLE); return path
for row in smoke:
    path=safe_path(row['audio_file'])
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==row['audio_sha256']
    assert normalize_raw_reference(row['reference_raw'])==row['reference']
    assert row['reference'].strip() and 0.5 <= float(row['duration_seconds']) < 30
# Seven shortest plus longest: eight distinct existing training components, including the duration boundary.
ordered=sorted(smoke,key=lambda r:(float(r['duration_seconds']),r['sample_id']))
selected=ordered[:7]+[ordered[-1]]
assert len({r['sample_id'] for r in selected})==8
print('Training-only examples:',len(selected),'duration range',min(float(r['duration_seconds']) for r in selected),max(float(r['duration_seconds']) for r in selected))


## 2. Pin the model and declare target/prefix policy

Training labels contain the exact normalized IPA text tokens **plus one EOS**. The model inserts its decoder-start token when shifting labels. We do **not** add an English token, a Nyishi token, a transcribe token, or a no-timestamps token to supervised labels. This explicitly differs from B0 automatic English selection, which remains a generation observation. Phase 6B must document this policy before comparing CER.


In [ ]:
MODEL_ID='neurlang/ipa-whisper-base'; REVISION='949f7c4e4ee374b5202bc5886f6e4552e5ae2371'
processor=WhisperProcessor.from_pretrained(MODEL_ID,revision=REVISION)
model=WhisperForConditionalGeneration.from_pretrained(MODEL_ID,revision=REVISION,dtype=torch.float32,attn_implementation='sdpa').to(DEVICE)
assert next(model.parameters()).dtype==torch.float32
model.train(); model.config.use_cache=False
assert model.config.decoder_start_token_id==processor.tokenizer.convert_tokens_to_ids('<|startoftranscript|>')
assert model.config.max_target_positions >= 448
EOS=processor.tokenizer.eos_token_id; START=model.config.decoder_start_token_id
assert EOS is not None and EOS!=START
assert model.generation_config.begin_suppress_tokens==[220,50257]
print({'revision':REVISION,'decoder_start':START,'eos':EOS,'max_target_positions':model.config.max_target_positions,
       'generation_language':getattr(model.generation_config,'language',None),
       'generation_task':getattr(model.generation_config,'task',None),
       'begin_suppress_tokens':model.generation_config.begin_suppress_tokens})
def normalize(text): return re.sub(r' {2,}',' ',unicodedata.normalize('NFD',text).strip())
features=[]; targets=[]; target_info=[]
for row in selected:
    waveform,sr=sf.read(safe_path(row['audio_file']),dtype='float32',always_2d=True)
    assert sr==16000 and waveform.shape[1]==1 and len(waveform)>0
    assert abs(len(waveform)/sr-float(row['duration_seconds'])) < 0.02
    raw=row['reference_raw']; ref=row['reference']
    assert normalize(raw)==ref==normalize(ref)
    tensor=processor.feature_extractor(waveform[:,0],sampling_rate=sr,return_tensors='pt').input_features[0]
    assert tuple(tensor.shape)==(80,3000) and tensor.dtype==torch.float32 and torch.isfinite(tensor).all()
    ids=processor.tokenizer(ref,add_special_tokens=False).input_ids
    assert ids and all(x!=EOS for x in ids)
    assert processor.tokenizer.decode(ids,skip_special_tokens=False)==ref, 'Tokenizer round trip failed; stop.'
    target=ids+[EOS]
    assert len(target)+1 <= model.config.max_target_positions, 'Start + text + EOS exceeds decoder capacity.'
    features.append(tensor); targets.append(target)
    target_info.append({'target_tokens_plus_eos':len(target),'reference_codepoints':len(ref)})
print('Target lengths plus EOS:',[x['target_tokens_plus_eos'] for x in target_info])


## 3. Collator and one real FP16 AMP optimizer update

Audio features and targets are padded separately. Only positions where the **target attention mask is zero** become `-100`; EOS stays a supervised target. Start with the sprint plan's micro-batch 2 and accumulation 4, giving one update from eight clips. If an out-of-memory error occurs, stop and rerun from a fresh session with micro-batch 1 and accumulation 8, recording that change. The smoke uses a constant one-step scheduler so both the initial and resumed updates can prove nonzero weight changes; the pilot's provisional warmup/decay schedule is a separate Phase 6B choice.


In [ ]:
MICRO_BATCH=2; ACCUMULATION=4
assert MICRO_BATCH*ACCUMULATION==8 and 8%MICRO_BATCH==0
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-5,weight_decay=0.01)
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lambda step:1.0)
scaler=torch.amp.GradScaler('cuda',init_scale=256.0)
assert scaler.is_enabled()
def make_batch(indices):
    x=torch.stack([features[i] for i in indices]).to(DEVICE)
    padded=processor.tokenizer.pad({'input_ids':[targets[i] for i in indices]},padding=True,return_tensors='pt')
    labels=padded['input_ids'].masked_fill(padded['attention_mask'].ne(1),-100)
    for j,i in enumerate(indices):
        assert labels[j,len(targets[i])-1].item()==EOS
        assert torch.all(labels[j,len(targets[i]):].eq(-100))
        assert labels[j,0].item()!=START
    return {'input_features':x,'labels':labels.to(DEVICE)}
batch=make_batch([0,1]); assert batch['input_features'].shape==(2,80,3000)
assert (batch['labels']==-100).any(), 'Expected a real padding position in smoke batch.'
print('Collator: independent feature/target padding; target mask verified; EOS retained.')
# A representative trainable decoder weight provides a direct update check.
probe=model.model.decoder.layers[0].self_attn.q_proj.weight
before=probe.detach().float().cpu().clone()
def finite_amp_update(active_model, active_optimizer, active_scaler, active_probe):
    # A nonfinite norm is a failed gate for this attempt; never clip it into an update.
    # Retry the same eight examples at a lower loss scale, with fresh gradients.
    diagnostics=[]
    for attempt in range(1,9):
        active_optimizer.zero_grad(set_to_none=True)
        attempt_losses=[]
        for offset in range(0,8,MICRO_BATCH):
            micro=make_batch(list(range(offset,offset+MICRO_BATCH)))
            with torch.autocast('cuda',dtype=torch.float16):
                loss=active_model(**micro).loss
            if not torch.isfinite(loss).item():
                raise RuntimeError('Nonfinite forward loss under FP16 AMP; Phase 6A gate failed.')
            attempt_losses.append(float(loss.detach()))
            active_scaler.scale(loss/ACCUMULATION).backward()
        active_scaler.unscale_(active_optimizer)  # once, after all accumulation
        if active_probe.grad is None:
            raise RuntimeError('Probe parameter has no gradient; Phase 6A gate failed.')
        nonfinite_params=sum(p.grad is not None and not torch.isfinite(p.grad).all().item()
                             for p in active_model.parameters())
        scale_used=float(active_scaler.get_scale())
        if nonfinite_params:
            # unscale_ recorded the overflow; update() lowers the scale and resets its step state.
            active_scaler.update()
            diagnostics.append({'attempt':attempt,'scale':scale_used,'nonfinite_gradient_parameters':nonfinite_params})
            continue
        norm=torch.nn.utils.clip_grad_norm_(active_model.parameters(),1.0,error_if_nonfinite=False,foreach=False)
        if not torch.isfinite(norm).item() or float(norm)<=0:
            active_scaler.update(new_scale=max(scale_used/2,1.0))
            diagnostics.append({'attempt':attempt,'scale':scale_used,'nonfinite_or_zero_global_norm':True})
            continue
        active_scaler.step(active_optimizer)
        active_scaler.update()
        if active_scaler.get_scale()<scale_used:
            diagnostics.append({'attempt':attempt,'scale':scale_used,'optimizer_step_skipped':True})
            continue
        return attempt_losses,norm,attempt,diagnostics
    raise RuntimeError(f'Phase 6A gate failed: no finite FP16 update after 8 attempts; diagnostics={diagnostics}')

start_time=time.perf_counter()
torch.cuda.reset_peak_memory_stats(DEVICE)
losses,grad_norm,first_attempts,first_amp_diagnostics=finite_amp_update(model,optimizer,scaler,probe)
scheduler.step()
change=float(torch.max(torch.abs(probe.detach().float().cpu()-before)))
assert change>0 and int(optimizer.state[probe]['step'].item())==1, 'No real optimizer update.'
first_elapsed=time.perf_counter()-start_time
first_peak=torch.cuda.max_memory_allocated(DEVICE)/(1024**2)
print({'losses':losses,'grad_norm_before_clip':float(grad_norm),'first_weight_max_change':change,
       'first_update_seconds':first_elapsed,'first_amp_attempts':first_attempts,
       'first_amp_diagnostics':first_amp_diagnostics,'first_update_peak_allocated_mib':first_peak,'scaler':scaler.get_scale()})


## 4. Save, reload, and continue one update

The checkpoint contains model weights, optimizer moment/state, scheduler state, AMP scaler, RNG states, and an explicit completed-update count. The reloaded model is built from saved weights after releasing the first model to avoid doubling GPU memory. A second update proves that the restored state is usable. Checkpoint files and per-clip labels are private.


In [ ]:
import gc
OUT=Path('/kaggle/working/phase6a_private'); OUT.mkdir(parents=True,exist_ok=True)
MODEL_DIR=OUT/'model_after_first_update'; MODEL_DIR.mkdir(exist_ok=True)
model.save_pretrained(MODEL_DIR,safe_serialization=True)
state={'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
       'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
       'cuda_rng':torch.cuda.get_rng_state_all(),'completed_updates':1,'seed':SEED,
       'model_revision':REVISION,'micro_batch':MICRO_BATCH,'accumulation':ACCUMULATION}
torch.save(state,OUT/'training_state.pt')
weight_after_first=probe.detach().cpu().clone()
del probe,optimizer,scheduler,scaler,model,batch
gc.collect(); torch.cuda.empty_cache()
resumed=WhisperForConditionalGeneration.from_pretrained(MODEL_DIR,dtype=torch.float32,attn_implementation='sdpa').to(DEVICE)
resumed.train(); resumed.config.use_cache=False
probe2=resumed.model.decoder.layers[0].self_attn.q_proj.weight
assert torch.equal(probe2.detach().cpu(),weight_after_first), 'Model weight reload differs.'
optimizer2=torch.optim.AdamW(resumed.parameters(),lr=1e-5,weight_decay=0.01)
scheduler2=torch.optim.lr_scheduler.LambdaLR(optimizer2,lr_lambda=lambda step:1.0)
scaler2=torch.amp.GradScaler('cuda',init_scale=256.0)
loaded=torch.load(OUT/'training_state.pt',map_location='cpu',weights_only=False)
assert loaded['completed_updates']==1 and loaded['model_revision']==REVISION
optimizer2.load_state_dict(loaded['optimizer'])
for item in optimizer2.state.values():
    for k,v in item.items():
        if isinstance(v,torch.Tensor): item[k]=v.to(DEVICE)
scheduler2.load_state_dict(loaded['scheduler']); scaler2.load_state_dict(loaded['scaler'])
random.setstate(loaded['python_rng']); np.random.set_state(loaded['numpy_rng'])
torch.set_rng_state(loaded['torch_rng']); torch.cuda.set_rng_state_all(loaded['cuda_rng'])
assert scheduler2.last_epoch==1
assert scaler2.get_scale()==loaded['scaler']['scale']
assert len(optimizer2.state)>0 and any('exp_avg' in x for x in optimizer2.state.values())
assert torch.equal(torch.get_rng_state(),loaded['torch_rng'])
assert all(torch.equal(a,b) for a,b in zip(torch.cuda.get_rng_state_all(),loaded['cuda_rng']))
before_resume=probe2.detach().cpu().clone()
torch.cuda.reset_peak_memory_stats(DEVICE)
resume_start=time.perf_counter()
resume_losses,resume_grad,resume_attempts,resume_amp_diagnostics=finite_amp_update(resumed,optimizer2,scaler2,probe2)
scheduler2.step()
resume_change=float(torch.max(torch.abs(probe2.detach().cpu()-before_resume)))
assert resume_change>0 and int(optimizer2.state[probe2]['step'].item())==2
resume_elapsed=time.perf_counter()-resume_start
resume_peak=torch.cuda.max_memory_allocated(DEVICE)/(1024**2)
print({'resume_losses':resume_losses,'resumed_weight_max_change':resume_change,
       'resumed_update_seconds':resume_elapsed,'resume_amp_attempts':resume_attempts,
       'resume_amp_diagnostics':resume_amp_diagnostics,'resumed_update_peak_allocated_mib':resume_peak})


## 5. Private report and gate

Review the generated JSON and keep the executed notebook, bundle, and checkpoint private. Phase 6A passes only if all assertions above passed. Training loss here is a pipeline diagnostic, not a validation CER result. Phase 6B remains a separate, component-aware 1–2 hour pilot with validation-based checkpoint selection.


In [ ]:
report={'phase':'6A','status':'passed','scope':'eight training-only smoke clips; two optimizer updates including one resumed update',
        'model_id':MODEL_ID,'revision':REVISION,'versions':{'python':platform.python_version(),'torch':torch.__version__,
        'transformers':transformers.__version__,'tokenizers':tokenizers.__version__,
        'hub':huggingface_hub.__version__,'safetensors':safetensors.__version__},
        'gpu':torch.cuda.get_device_name(0),'seed':SEED,'precision':'FP16 autocast; FP32 model and optimizer',
        'micro_batch':MICRO_BATCH,'accumulation':ACCUMULATION,'effective_batch':8,'optimizer':'AdamW 1e-5, weight_decay 0.01',
        'scheduler':'constant for 6A resume/update gate only','gradient_checkpointing':False,
        'target_policy':'NFD IPA text tokens + one EOS; decoder start inserted by model; no language/task token in labels',
        'generation_policy':'B0 unforced generation retained; no validation/test generation in 6A',
        'target_lengths_plus_eos':[x['target_tokens_plus_eos'] for x in target_info],
        'first_losses':losses,'resumed_losses':resume_losses,
        'first_amp_attempts':first_attempts,'resume_amp_attempts':resume_attempts,
        'first_amp_diagnostics':first_amp_diagnostics,'resume_amp_diagnostics':resume_amp_diagnostics,
        'first_weight_max_change':change,'resumed_weight_max_change':resume_change,
        'first_update_seconds':first_elapsed,'resume_update_seconds':resume_elapsed,
        'first_peak_allocated_mib':first_peak,'resume_peak_allocated_mib':resume_peak,
        'checkpoint_has_optimizer_scheduler_scaler_rng':True,
        'private_bundle_sha256':hashlib.sha256((BUNDLE/'SHA256SUMS.txt').read_bytes()).hexdigest(),
        'test_data_used':False,'validation_cer_computed':False}
(OUT/'phase6a_report_private.json').write_text(json.dumps(report,indent=2)+'\n')
print('PHASE 6A GATE PASSED. Private report:',OUT/'phase6a_report_private.json')
